# RAG with Feast

#### Install required dependencies

In [ ]:
%pip install --quiet pymilvus feast transformers sentence-transformers faiss-cpu datasets
%pip install --upgrade protobuf

#### Retrieve test dataset and chunk it

In [ ]:
import urllib.request

link = "https://huggingface.co/ngxson/demo_simple_rag_py/raw/main/cat-facts.txt"
dataset = []

# Retrieve knowledge from provided link, use every line as a separate chunk.
for line in urllib.request.urlopen(link):
    dataset.append(line.decode('utf-8'))

print(f'Loaded {len(dataset)} entries')

#### Define models and generate embeddings

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

# load pretrained model sentence transformer model
embedding_model = SentenceTransformer("ibm-granite/granite-embedding-30m-english")
embeddings = embedding_model.encode(dataset)

print(f"Generated embeddings of shape: {embeddings.shape}")

question_encoder_tokenizer = embedding_model.tokenizer

generator_model_id = "ibm-granite/granite-3.2-2b-instruct"
generator_model = AutoModelForCausalLM.from_pretrained(generator_model_id)
generator_tokenizer = AutoTokenizer.from_pretrained(generator_model_id)


#### Connect to Milvus vector database

In [ ]:
from pymilvus import connections

# Connect to milvus 
connections.connect(alias="default", host="vectordb-milvus.milvus.svc.cluster.local", port="19530", timeout=10, user="root", password="Milvus")

# Check connection status 
print("Connected:", connections.has_connection(alias="default"))

#### Create collection in Milvus

In [ ]:
from pymilvus import FieldSchema, CollectionSchema, DataType, Collection, utility

# Drop the old collection if it exists
if utility.has_collection("cat_facts"):
    Collection("cat_facts", using="default").drop()

# Define schema
fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=False),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=384)
]
schema = CollectionSchema(fields)
collection = Collection(name="cat_facts", schema=schema, using="default")

# Create index
collection.create_index(
    field_name="embedding",
    index_params={"index_type": "IVF_FLAT", "metric_type": "COSINE", "params": {"nlist": 64}}
)

collection.load()

#### Insert embeddings into Milvus

In [ ]:
ids = list(range(len(dataset)))
vectors = embeddings.tolist()

collection.insert([ids, vectors])
collection.flush()

print(f"Inserted {len(ids)} cat fact embeddings into Milvus.")

#### Initialise Feature Store (Feast)

In [ ]:
!feast init catproject

#### Using test dataset to create parquet file as historical data source

In [ ]:
import pandas as pd
from datetime import datetime

# Create DataFrame
df = pd.DataFrame({
    "cat_fact_id": list(range(len(dataset))),
    "fact": dataset,
    "embedding": pd.Series(list(embeddings), dtype=object),
    "event_timestamp": [datetime.utcnow() for _ in dataset],
})

# Save to Parquet
df.to_parquet("catproject/feature_repo/data/cat_facts.parquet", index=False)
print("Saved to cat_facts.parquet")

#### Move into catproject directory and create feast objects

In [ ]:
%cd catproject/feature_repo
!feast apply

#### Initialize Milvus Vector Store and FeastRAGRetriever

In [ ]:
from transformers import RagConfig, AutoConfig
from feast_rag_retriever import MilvusVectorStore, FeastRAGRetriever

# Initialize Milvus Vector Store
milvus_store = MilvusVectorStore(collection_name="cat_facts")

generator_subconfig = AutoConfig.from_pretrained(generator_model_id)
question_encoder_subconfig = AutoConfig.from_pretrained(generator_model_id)

generator_config = RagConfig(
    index_name="custom",
    retrieval_vector_size=384,
    question_encoder=question_encoder_subconfig.to_dict(),
    generator=generator_subconfig.to_dict(),
    # index_path=None,
    # passages_path=None,
)

# Initialize the FeastRAGRetriever
rag_retriever = FeastRAGRetriever(
    question_encoder_tokenizer=question_encoder_tokenizer,
    question_encoder=embedding_model,
    generator_tokenizer=generator_tokenizer,
    feast_repo_path=".",
    generator_model=generator_model,
    vector_store=milvus_store,
    search_type="vector",
    config=generator_config,
    index=None,
)

#### Retrieve top-k documents

In [ ]:
top_k_documents = rag_retriever.retrieve(embeddings, top_k=5)

#### Generate response for provided query using top_k retrieved context

In [ ]:
query = "Do cats like to climb?"
answer = rag_retriever.generate_answer(query, top_k=5, max_new_tokens=100)

print("Query:", query)
print("Answer:", answer)